# XAS from raw H5

Build a single-scan XAS measurement from one data run by:

1. Loading a separate **dark run** (x-rays off) and computing a per-bunch
   2D background from its mean VLS spectrum.
2. Loading the **data run** (x-rays on), cropping to the ROI, and
   subtracting the dark-run background.
3. Optionally subtracting an additional 1D background built from the
   end-of-train bunches in the data run itself, to soak up any drift
   between the dark acquisition and the data acquisition.
4. Aligning vls and gmd by train ID (handled automatically by
   `load_raw_h5`).
5. Checking the per-shot correlation between the integrated VLS
   intensity and the GMD pulse energy.
6. Computing the scalar `XAS = sum(gmd) / sum(vls)` over every shot in
   the signal bunch range — one number for the scan at its mean
   photon energy.
7. Binning by central photon energy `mpe` to draw an XAS curve across
   the energy range the scan covered (use this to see absorption
   edges).

In [ ]:
import sys
from pathlib import Path
_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import config
from data_loading import load_raw_h5
from binning import arb_bool_ar, bin_and_sum_ratio

%matplotlib inline

## Parameters

In [ ]:
# Runs --------------------------------------------------------------
DARK_RUN_NO = 58763   # x-rays off (dark)
DATA_RUN_NO = 58764   # x-rays on  (signal)
MAX_FILES   = 5       # raw H5 files per run; None = read all (heavy!)

# VLS pixel-space ROI around the absorption edge.
ROI = (500, 600)

# Bunches that actually saw x-rays in the data run (half-open).
# Verify against the per-bunch diagnostic plot below.
SIGNAL_BUNCH_RANGE = (0, 60)

# Optional additional 1D background built from end-of-train "off"
# bunches in the data run. Set to None to skip this step.
BG_BUNCH_RANGE = (90, 100)

# Per-shot GMD band (uJ). None = no bound on that side.
GMD_LO, GMD_HI = 0.5, None

# Number of mpe bins for the XAS curve.
N_MPE_BINS = 10

## 1. Dark run -> per-bunch background

Load the dark run, crop the VLS to the ROI, and average over all trains
to get a 2D `(n_bunches, n_pixels)` background. Keeping the bunch axis
lets a bunch-dependent detector baseline be removed too — which a
single 1D spectrum would smear out.

In [ ]:
dark = load_raw_h5(DARK_RUN_NO, config=2, max_files=MAX_FILES)
dark = dark.crop_vls(*ROI)
dark_bg = np.nanmean(dark.vls, axis=0)   # (n_bunches, n_pixels)
print(f"dark vls          : {dark.vls.shape}")
print(f"dark_bg           : {dark_bg.shape}  (mean over {dark.n_trains} trains)")

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.pcolormesh(dark.vls_pixels, np.arange(dark_bg.shape[0]), dark_bg,
                   cmap="inferno", shading="auto")
fig.colorbar(im, ax=ax, label="Dark intensity (arb.)")
ax.set_xlabel("Pixel")
ax.set_ylabel("Bunch index")
ax.set_title(f"Per-bunch dark background  (run {DARK_RUN_NO})")
fig.tight_layout()
plt.show()

## 2. Data run -> subtract dark + optional trailing-bunches bg

After the dark subtraction, the signal-free trailing bunches
(`BG_BUNCH_RANGE`) should sit near zero. If they don't, that's residual
drift between the dark and data acquisitions — the optional 1D
`auto_subtract_background` step soaks it up.

In [ ]:
data = load_raw_h5(DATA_RUN_NO, config=2, max_files=MAX_FILES)
data = data.crop_vls(*ROI)
print(f"data vls          : {data.vls.shape}")

if data.n_bunches != dark_bg.shape[0]:
    raise ValueError(
        f"bunch count mismatch: data has {data.n_bunches} bunches, "
        f"dark_bg has {dark_bg.shape[0]}. Pass a matching train_length "
        f"to load_raw_h5 for both runs."
    )

data = data.subtract_background(dark_bg)
print(f"after dark sub    : mean={float(np.nanmean(data.vls)):.2f}")

if BG_BUNCH_RANGE is not None:
    data = data.auto_subtract_background(BG_BUNCH_RANGE)
    print(f"after auto sub    : mean={float(np.nanmean(data.vls)):.2f}  "
          f"(BG_BUNCH_RANGE={BG_BUNCH_RANGE})")

## 3. Mean spectrum vs bunch index

Cyan dashes mark `SIGNAL_BUNCH_RANGE`. Use the right-hand
integrated-intensity panel to confirm signal is contained in the
signal range and that the background range is in the dark tail.

In [ ]:
mean_by_bunch  = np.nanmean(data.vls, axis=0)              # (n_bunches, n_pixels)
total_by_bunch = np.nansum(mean_by_bunch, axis=1)          # (n_bunches,)
n_bunches = data.n_bunches
b_start, b_end = SIGNAL_BUNCH_RANGE

fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
im = axes[0].pcolormesh(data.vls_pixels, np.arange(n_bunches), mean_by_bunch,
                        cmap="inferno", shading="auto")
fig.colorbar(im, ax=axes[0], label="Mean intensity (arb.)")
axes[0].axhline(b_start, color="cyan", lw=1, ls="--")
axes[0].axhline(b_end,   color="cyan", lw=1, ls="--")
axes[0].set_xlabel("Pixel")
axes[0].set_ylabel("Bunch index")
axes[0].set_title(f"Data mean spectrum vs bunch  (run {DATA_RUN_NO})")

axes[1].plot(total_by_bunch, np.arange(n_bunches), "o-", ms=3,
             color="mediumseagreen")
axes[1].axhline(b_start, color="cyan", lw=1, ls="--", label="signal range")
axes[1].axhline(b_end,   color="cyan", lw=1, ls="--")
if BG_BUNCH_RANGE is not None:
    axes[1].axhline(BG_BUNCH_RANGE[0], color="orange", lw=1, ls=":",
                    label="bg range")
    axes[1].axhline(BG_BUNCH_RANGE[1], color="orange", lw=1, ls=":")
axes[1].set_xlabel("Integrated intensity")
axes[1].set_title("per bunch")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower right")
fig.tight_layout()
plt.show()

## 4. Per-shot quantities

Flatten over (train, signal bunch) into per-shot vectors:

- `gmd_shot`     — per-bunch incident pulse energy
- `vls_sum_shot` — integrated VLS intensity over the pixel ROI
- `mpe_shot`     — central photon energy of the train, broadcast to bunches

`good` keeps only finite shots inside the optional `[GMD_LO, GMD_HI]`
band; all subsequent per-shot operations index through it so the
correlation, scalar XAS, and binned XAS all use the same population.

In [ ]:
b_start, b_end = SIGNAL_BUNCH_RANGE
m_sig = b_end - b_start

vls_sig = data.vls[:, b_start:b_end, :]                  # (n, m_sig, n_px)
gmd_sig = data.gmd[:, b_start:b_end]                     # (n, m_sig)
vls_sum_shot = np.nansum(vls_sig, axis=-1).ravel()       # (n*m_sig,)
gmd_shot     = gmd_sig.ravel()
mpe_shot     = np.broadcast_to(data.mpe[:, None],
                               (data.n_trains, m_sig)).ravel()

good = np.isfinite(gmd_shot) & np.isfinite(vls_sum_shot) & np.isfinite(mpe_shot)
if GMD_LO is not None: good &= gmd_shot >= GMD_LO
if GMD_HI is not None: good &= gmd_shot <= GMD_HI

x_gmd = gmd_shot[good]
y_vls = vls_sum_shot[good]
mpe   = mpe_shot[good]
print(f"shots total       : {gmd_shot.size}")
print(f"shots after filter: {x_gmd.size}")
print(f"mpe range         : {np.nanmin(mpe):.2f} .. {np.nanmax(mpe):.2f} eV")

## 5. Correlation: GMD vs integrated VLS

A clean linear correlation through the origin means dark/background
subtraction is reasonable and the train-ID alignment between the GMD
and the VLS is intact.

In [ ]:
gmd_edges = np.percentile(x_gmd, np.linspace(0, 100, 11))
gmd_cents, gmd_bool_ar = arb_bool_ar(gmd_edges, x_gmd)

vls_mean_per_gmd = np.array([
    np.nanmean(y_vls[m]) if m.any() else np.nan for m in gmd_bool_ar
])
vls_std_per_gmd  = np.array([
    np.nanstd(y_vls[m])  if m.any() else np.nan for m in gmd_bool_ar
])
gmd_mean_per_gmd = np.array([
    np.nanmean(x_gmd[m]) if m.any() else np.nan for m in gmd_bool_ar
])

r = float(np.corrcoef(x_gmd, y_vls)[0, 1])
slope, intercept = np.polyfit(x_gmd, y_vls, 1)
xl = np.linspace(gmd_mean_per_gmd.min(), gmd_mean_per_gmd.max(), 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
h = ax1.hist2d(
    x_gmd, y_vls,
    bins=[gmd_edges,
          np.linspace(np.percentile(y_vls, 1), np.percentile(y_vls, 99), 51)],
    cmap="viridis", cmin=1, norm=mcolors.LogNorm(),
)
fig.colorbar(h[3], ax=ax1, label="shots per bin")
ax1.plot(xl, slope*xl + intercept, color="white", lw=1.5,
         label=f"slope={slope:.3g}, intercept={intercept:.3g}, r={r:.3f}")
ax1.set_xlabel("GMD (uJ)")
ax1.set_ylabel("Integrated VLS (arb.)")
ax1.set_title(f"GMD vs integrated VLS  ({x_gmd.size} shots)")
ax1.legend(loc="upper left", framealpha=0.85)

ax2.errorbar(gmd_mean_per_gmd, vls_mean_per_gmd, yerr=vls_std_per_gmd,
             fmt="o-", color="mediumseagreen", capsize=3,
             label="binned mean +/- std")
ax2.plot(xl, slope*xl + intercept, color="darkred", lw=1.2,
         label="linear fit (all shots)")
ax2.set_xlabel("GMD (uJ)")
ax2.set_ylabel("Mean integrated VLS")
ax2.set_title("Percentile-binned")
ax2.grid(alpha=0.3)
ax2.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 6. Scalar XAS for this scan

`XAS = sum(gmd) / sum(vls)` over every kept shot. One number — the
absorption metric at this scan's mean photon energy. Energy-normalised
form, per the project convention (sum/sum, not mean(x/y)).

In [ ]:
gmd_total = float(np.nansum(x_gmd))
vls_total = float(np.nansum(y_vls))
xas_scalar = gmd_total / vls_total
print(f"sum(gmd)                       : {gmd_total:.3e}")
print(f"sum(vls)                       : {vls_total:.3e}")
print(f"XAS = sum(gmd) / sum(vls)      : {xas_scalar:.4f}")
print(f"<mpe> over kept shots          : {float(np.nanmean(mpe)):.2f} eV")

## 7. XAS vs central photon energy (`mpe` binning)

Each train has an upstream mean photon energy `mpe`. Binning the
kept shots by `mpe` and computing `sum(gmd) / sum(vls)` per bin gives
an XAS curve across whatever range the scan covered. Use percentile
bins so each point on the curve is averaged over the same number of
shots.

In [ ]:
mpe_edges = np.percentile(mpe, np.linspace(0, 100, N_MPE_BINS + 1))
_, mpe_bool_ar = arb_bool_ar(mpe_edges, mpe)

# Per-bin XAS = sum(gmd) / sum(vls) — the canonical sum/sum pattern.
xas_per_bin, n_per_bin = bin_and_sum_ratio(mpe_bool_ar, x_gmd, y_vls)

# Bin centre on the mpe axis = weighted-by-count mean of mpe in the bin.
mpe_mean_per_bin = np.array([
    np.nanmean(mpe[m]) if m.any() else np.nan for m in mpe_bool_ar
])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mpe_mean_per_bin, xas_per_bin, "o-",
        color="mediumseagreen", lw=1.5)
ax.set_xlabel("Mean photon energy per bin (eV)")
ax.set_ylabel("XAS = sum(gmd) / sum(vls)")
ax.set_title(f"XAS vs mpe  (run {DATA_RUN_NO}, {N_MPE_BINS} percentile bins)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"{'mpe (eV)':>12s}  {'n shots':>8s}  {'XAS':>10s}")
for c, n, v in zip(mpe_mean_per_bin, n_per_bin, xas_per_bin):
    print(f"{c:12.2f}  {int(n):8d}  {v:10.4f}")